In [33]:
import pandas as pd

In [34]:
# 1. Ruta relativa desde la carpeta notebooks hacia los archivos CSV
ruta_competencia = '../data/raw/entrenamiento/competencia.csv'
ruta_ventas = '../data/raw/entrenamiento/ventas.csv'

# 2. Cargar los archivos CSV en sus respectivos DataFrames
competencia_df = pd.read_csv(ruta_competencia, parse_dates=['fecha'])
ventas_df = pd.read_csv(ruta_ventas, parse_dates=['fecha'])

# 3. Mostrar las primeras filas para verificar la carga
print('Primeras filas de competencia:')
print(competencia_df.head())
print('\nPrimeras filas de ventas:')
print(ventas_df.head())

Primeras filas de competencia:
       fecha producto_id  Amazon  Decathlon  Deporvillage
0 2021-10-25    PROD_001   82.96     111.88         97.43
1 2021-10-25    PROD_002  112.56     108.61        115.58
2 2021-10-25    PROD_003   79.79      78.44         80.11
3 2021-10-25    PROD_004   72.60      67.29         74.45
4 2021-10-25    PROD_005   37.71      33.60         33.07

Primeras filas de ventas:
       fecha producto_id                            nombre categoria  \
0 2021-10-25    PROD_001          Nike Air Zoom Pegasus 40   Running   
1 2021-10-25    PROD_002              Adidas Ultraboost 23   Running   
2 2021-10-25    PROD_003               Asics Gel Nimbus 25   Running   
3 2021-10-25    PROD_004  New Balance Fresh Foam X 1080v12   Running   
4 2021-10-25    PROD_005                Nike Dri-FIT Miler   Running   

         subcategoria  precio_base  es_estrella  unidades_vendidas  \
0  Zapatillas Running          115         True                  6   
1  Zapatillas Running

In [35]:
# 4. Revisar la estructura, los tipos y los valores nulos
print('Dimensiones:', ventas_df.shape)
print('\nTipos de datos por variable:')
print(ventas_df.dtypes)
print('\nValores nulos por variable:')
print(ventas_df.isna().sum())

# 5. Revisar registros duplicados y la unicidad fecha-producto
print('\nFilas completamente duplicadas:', ventas_df.duplicated().sum())
print('Claves fecha-producto duplicadas:', ventas_df.duplicated(subset=['fecha', 'producto_id']).sum())

# 6. Validar fechas, cobertura por fecha y consistencia de ingresos
fechas_validas = pd.to_datetime(ventas_df['fecha'], errors='coerce')
productos_esperados = ventas_df['producto_id'].nunique()
productos_por_fecha = ventas_df.groupby('fecha')['producto_id'].nunique()
ingresos_calculados = ventas_df['unidades_vendidas'] * ventas_df['precio_venta']
inconsistencias_ingresos = (ventas_df['ingresos'] - ingresos_calculados).abs() > 0.01
print('\nFechas no convertibles:', fechas_validas.isna().sum())
print('Rango de fechas:', fechas_validas.min(), 'a', fechas_validas.max())
print('Fechas sin todos los productos:', (productos_por_fecha != productos_esperados).sum())
print('Ingresos distintos de unidades x precio de venta:', inconsistencias_ingresos.sum())
ventas_sobre_precio_base = ventas_df['precio_venta'] > ventas_df['precio_base']
print('Precios de venta superiores al precio base:', ventas_sobre_precio_base.sum())

# 7. Validar valores numéricos no positivos
print('\nValores no positivos por variable:')
print((ventas_df[['precio_base', 'unidades_vendidas', 'precio_venta', 'ingresos']] <= 0).sum())

# 8. Revisar descriptivos numéricos y categóricos
print('\nEstadísticos descriptivos:')
print(ventas_df.describe(include='all').T.to_string())

# 9. Contar posibles valores atípicos con el criterio IQR (para revisión, no implica error)
for variable in ['precio_base', 'unidades_vendidas', 'precio_venta', 'ingresos']:
    cuartil_1 = ventas_df[variable].quantile(0.25)
    cuartil_3 = ventas_df[variable].quantile(0.75)
    rango_iqr = cuartil_3 - cuartil_1
    limite_inferior = cuartil_1 - 1.5 * rango_iqr
    limite_superior = cuartil_3 + 1.5 * rango_iqr
    valores_atipicos = ((ventas_df[variable] < limite_inferior) | (ventas_df[variable] > limite_superior)).sum()
    print(f'{variable}: {valores_atipicos} posibles valores atípicos')

# 10. Emitir el dictamen general de calidad
hay_nulos = ventas_df.isna().any().any()
hay_duplicados = ventas_df.duplicated().any() or ventas_df.duplicated(subset=['fecha', 'producto_id']).any()
hay_fechas_invalidas = fechas_validas.isna().any()
hay_ingresos_inconsistentes = inconsistencias_ingresos.any()
hay_no_positivos = (ventas_df[['precio_base', 'unidades_vendidas', 'precio_venta', 'ingresos']] <= 0).any().any()
if hay_nulos or hay_duplicados or hay_fechas_invalidas or hay_ingresos_inconsistentes or hay_no_positivos:
    print('\nDictamen: se detectaron incidencias que requieren revisión.')
else:
    print('\nDictamen: sin incidencias estructurales en nulos, duplicados, fechas, valores positivos ni cálculo de ingresos.')
print('Nota: fecha se carga como datetime y está lista para el análisis temporal.')
print('Nota: los valores atípicos IQR y precios de venta superiores al precio base requieren contexto de negocio; no son errores automáticamente.')

Dimensiones: (3552, 10)

Tipos de datos por variable:
fecha                datetime64[us]
producto_id                     str
nombre                          str
categoria                       str
subcategoria                    str
precio_base                   int64
es_estrella                    bool
unidades_vendidas             int64
precio_venta                float64
ingresos                    float64
dtype: object

Valores nulos por variable:
fecha                0
producto_id          0
nombre               0
categoria            0
subcategoria         0
precio_base          0
es_estrella          0
unidades_vendidas    0
precio_venta         0
ingresos             0
dtype: int64

Filas completamente duplicadas: 0
Claves fecha-producto duplicadas: 0

Fechas no convertibles: 0
Rango de fechas: 2021-10-25 00:00:00 a 2024-11-30 00:00:00
Fechas sin todos los productos: 0
Ingresos distintos de unidades x precio de venta: 0
Precios de venta superiores al precio base: 1567

Valores

In [36]:
# 11. Revisar la estructura, los tipos y los valores nulos de competencia_df
print('Dimensiones:', competencia_df.shape)
print('\nTipos de datos por variable:')
print(competencia_df.dtypes)
print('\nValores nulos por variable:')
print(competencia_df.isna().sum())

# 12. Revisar duplicados y unicidad de la clave fecha-producto
print('\nFilas completamente duplicadas:', competencia_df.duplicated().sum())
print('Claves fecha-producto duplicadas:', competencia_df.duplicated(subset=['fecha', 'producto_id']).sum())

# 13. Validar fechas, cobertura de productos y correspondencia con ventas_df
fechas_competencia_validas = pd.to_datetime(competencia_df['fecha'], errors='coerce')
productos_competencia_esperados = competencia_df['producto_id'].nunique()
productos_competencia_por_fecha = competencia_df.groupby('fecha')['producto_id'].nunique()
claves_competencia = set(zip(competencia_df['fecha'], competencia_df['producto_id']))
claves_ventas = set(zip(ventas_df['fecha'], ventas_df['producto_id']))
print('\nFechas no convertibles:', fechas_competencia_validas.isna().sum())
print('Rango de fechas:', fechas_competencia_validas.min(), 'a', fechas_competencia_validas.max())
print('Fechas sin todos los productos:', (productos_competencia_por_fecha != productos_competencia_esperados).sum())
print('Claves de competencia sin correspondencia en ventas_df:', len(claves_competencia - claves_ventas))
print('Claves de ventas_df sin correspondencia en competencia_df:', len(claves_ventas - claves_competencia))

# 14. Revisar descriptivos y precios no positivos de cada competidor
columnas_precios = ['Amazon', 'Decathlon', 'Deporvillage']
print('\nEstadísticos descriptivos:')
print(competencia_df.describe(include='all').T.to_string())
print('\nPrecios no positivos por competidor:')
print((competencia_df[columnas_precios] <= 0).sum())

# 15. Contar posibles valores atípicos IQR por competidor
for competidor in columnas_precios:
    cuartil_1 = competencia_df[competidor].quantile(0.25)
    cuartil_3 = competencia_df[competidor].quantile(0.75)
    rango_iqr = cuartil_3 - cuartil_1
    limite_inferior = cuartil_1 - 1.5 * rango_iqr
    limite_superior = cuartil_3 + 1.5 * rango_iqr
    valores_atipicos = ((competencia_df[competidor] < limite_inferior) | (competencia_df[competidor] > limite_superior)).sum()
    print(f'{competidor}: {valores_atipicos} posibles valores atípicos IQR')

# 16. Emitir el dictamen general de calidad
hay_nulos_competencia = competencia_df.isna().any().any()
hay_duplicados_competencia = competencia_df.duplicated().any() or competencia_df.duplicated(subset=['fecha', 'producto_id']).any()
hay_fechas_competencia_invalidas = fechas_competencia_validas.isna().any()
hay_precios_competencia_no_positivos = (competencia_df[columnas_precios] <= 0).any().any()
hay_claves_sin_correspondencia = claves_competencia != claves_ventas
if hay_nulos_competencia or hay_duplicados_competencia or hay_fechas_competencia_invalidas or hay_precios_competencia_no_positivos or hay_claves_sin_correspondencia:
    print('\nDictamen: se detectaron incidencias estructurales que requieren revisión.')
else:
    print('\nDictamen: sin incidencias estructurales en nulos, duplicados, fechas, precios positivos ni correspondencia con ventas_df.')
print('Nota: fecha se carga como datetime y está lista para el análisis temporal.')
print('Nota: los valores atípicos IQR deben revisarse por producto; el análisis global mezcla productos con distintos niveles de precio.')

Dimensiones: (3552, 5)

Tipos de datos por variable:
fecha           datetime64[us]
producto_id                str
Amazon                 float64
Decathlon              float64
Deporvillage           float64
dtype: object

Valores nulos por variable:
fecha           0
producto_id     0
Amazon          0
Decathlon       0
Deporvillage    0
dtype: int64

Filas completamente duplicadas: 0
Claves fecha-producto duplicadas: 0

Fechas no convertibles: 0
Rango de fechas: 2021-10-25 00:00:00 a 2024-11-30 00:00:00
Fechas sin todos los productos: 0
Claves de competencia sin correspondencia en ventas_df: 0
Claves de ventas_df sin correspondencia en competencia_df: 0

Estadísticos descriptivos:
               count unique       top freq                 mean                  min                  25%                  50%                  75%                  max         std
fecha           3552    NaN       NaN  NaN  2023-05-13 18:00:00  2021-10-25 00:00:00  2022-08-03 18:00:00  2023-05-13 12:00:00 

In [37]:
ventas_df.info

<bound method DataFrame.info of           fecha producto_id                            nombre categoria  \
0    2021-10-25    PROD_001          Nike Air Zoom Pegasus 40   Running   
1    2021-10-25    PROD_002              Adidas Ultraboost 23   Running   
2    2021-10-25    PROD_003               Asics Gel Nimbus 25   Running   
3    2021-10-25    PROD_004  New Balance Fresh Foam X 1080v12   Running   
4    2021-10-25    PROD_005                Nike Dri-FIT Miler   Running   
...         ...         ...                               ...       ...   
3547 2024-11-30    PROD_020                     Quechua MH500   Outdoor   
3548 2024-11-30    PROD_021              Manduka PRO Yoga Mat  Wellness   
3549 2024-11-30    PROD_022          Gaiam Premium Yoga Block  Wellness   
3550 2024-11-30    PROD_023                  Liforme Yoga Pad  Wellness   
3551 2024-11-30    PROD_024          Lotuscrafts Yoga Bolster  Wellness   

            subcategoria  precio_base  es_estrella  unidades_vendid

In [38]:
competencia_df.info

<bound method DataFrame.info of           fecha producto_id    Amazon  Decathlon  Deporvillage
0    2021-10-25    PROD_001   82.9600     111.88       97.4300
1    2021-10-25    PROD_002  112.5600     108.61      115.5800
2    2021-10-25    PROD_003   79.7900      78.44       80.1100
3    2021-10-25    PROD_004   72.6000      67.29       74.4500
4    2021-10-25    PROD_005   37.7100      33.60       33.0700
...         ...         ...       ...        ...           ...
3547 2024-11-30    PROD_020   86.2000      72.75       82.6800
3548 2024-11-30    PROD_021  141.7125     120.60      132.1875
3549 2024-11-30    PROD_022   20.3700      17.62       20.2900
3550 2024-11-30    PROD_023   32.6400      34.06       33.2300
3551 2024-11-30    PROD_024   46.5800      45.58       50.0800

[3552 rows x 5 columns]>

In [39]:
# 17. Integrar ventas y precios de la competencia usando fecha y producto_id como claves
df = pd.merge(
    ventas_df,
    competencia_df,
    on=['fecha', 'producto_id'],
    how='inner',
    validate='one_to_one'
)

# Verificar las dimensiones y las primeras filas del DataFrame integrado
print('Dimensiones de df:', df.shape)
print(df.head())

Dimensiones de df: (3552, 13)
       fecha producto_id                            nombre categoria  \
0 2021-10-25    PROD_001          Nike Air Zoom Pegasus 40   Running   
1 2021-10-25    PROD_002              Adidas Ultraboost 23   Running   
2 2021-10-25    PROD_003               Asics Gel Nimbus 25   Running   
3 2021-10-25    PROD_004  New Balance Fresh Foam X 1080v12   Running   
4 2021-10-25    PROD_005                Nike Dri-FIT Miler   Running   

         subcategoria  precio_base  es_estrella  unidades_vendidas  \
0  Zapatillas Running          115         True                  6   
1  Zapatillas Running          135         True                 10   
2  Zapatillas Running           85        False                  2   
3  Zapatillas Running           75        False                  2   
4        Ropa Running           35        False                  2   

   precio_venta  ingresos  Amazon  Decathlon  Deporvillage  
0        118.36    710.16   82.96     111.88         97

In [40]:
df.head

<bound method NDFrame.head of           fecha producto_id                            nombre categoria  \
0    2021-10-25    PROD_001          Nike Air Zoom Pegasus 40   Running   
1    2021-10-25    PROD_002              Adidas Ultraboost 23   Running   
2    2021-10-25    PROD_003               Asics Gel Nimbus 25   Running   
3    2021-10-25    PROD_004  New Balance Fresh Foam X 1080v12   Running   
4    2021-10-25    PROD_005                Nike Dri-FIT Miler   Running   
...         ...         ...                               ...       ...   
3547 2024-11-30    PROD_020                     Quechua MH500   Outdoor   
3548 2024-11-30    PROD_021              Manduka PRO Yoga Mat  Wellness   
3549 2024-11-30    PROD_022          Gaiam Premium Yoga Block  Wellness   
3550 2024-11-30    PROD_023                  Liforme Yoga Pad  Wellness   
3551 2024-11-30    PROD_024          Lotuscrafts Yoga Bolster  Wellness   

            subcategoria  precio_base  es_estrella  unidades_vendidas